# 06 — Avaliação: o fine-tuning funcionou?

**Entrada:** `../data/val.jsonl` e o adapter em `../modelos/lora_model`
**Saída:** os dois números que sustentam a seção de resultados do relatório

O `05` termina com uma loss de validação, mas **um número sozinho não diz nada**. Loss é
entropia cruzada sobre um tokenizer e um dataset específicos — 1,32 aqui não é comparável
a 1,32 em outro lugar. Só faz sentido contra uma referência.

Este notebook responde duas perguntas:

1. **O fine-tuning melhorou o modelo?** Comparando com o modelo base, sem LoRA, na mesma
   validação e com o mesmo cálculo de loss.
2. **Ele aprendeu conteúdo ou estilo?** Quebrando a loss entre as respostas de médico
   preservadas (15,4%) e as reescritas por LLM (84,6%). Se o modelo for muito melhor nas
   reescritas, aprendeu a imitar o `glm-5.3-flash`, não medicina.

> **Rode com o `05` parado.** Os dois disputam a mesma GPU. Reinicie o kernel do `05`
> antes de começar aqui.

In [1]:
import unsloth   # precisa vir antes de trl/transformers
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from pathlib import Path
import gc, torch, json

# Espelha o 05. Se mudar la, mude aqui -- senao os numeros nao sao comparaveis.
MODELO_BASE = "unsloth/Qwen3.5-4B"
ADAPTER = "../modelos/lora_model"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
SEED = 3407

DATA = Path("../data")
SAIDA = Path("../modelos")
SPLIT = "val"      # troque para "test" no numero final do relatorio

ds = load_dataset("json", data_files={SPLIT: str(DATA / f"{SPLIT}.jsonl")})[SPLIT]
humano = 1 - sum(ds["resposta_reescrita"]) / len(ds)
print(f"{SPLIT}: {len(ds):,} linhas | alvo humano {humano:.1%} | reescrito {1-humano:.1%}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/emidiosouza/FiapCode/fase3/TechV3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0910 18:16:01.312000 169837 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0910 18:16:01.358000 169837 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/home/emidiosouza/FiapCode/fase3/TechV3/.venv/lib/python3.12/site-packages/unsloth/import_fixes.py:2124: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


val: 1,530 linhas | alvo humano 15.9% | reescrito 84.1%


## Um modelo só, com o LoRA ligado e desligado

Carregar o modelo base sozinho não funciona: em 4 bits ele não pode ser treinado, e o
`Trainer` confere isso na hora em que é criado — antes de saber que só vamos avaliar.
Daí o `ValueError: You cannot perform fine-tuning on purely quantized models`.

Então carregamos só o fine-tunado e medimos duas vezes: uma normal, outra dentro de
`with model.disable_adapter():`, onde o LoRA é ignorado e sobra o modelo base. Além de
resolver o erro, os dois números saem do mesmo modelo e da mesma tokenização — não sobra
nenhuma diferença sem explicação. E carrega 4B de parâmetros uma vez em vez de duas.

**Detalhe que quebra se feito de outro jeito:** o `SFTTrainer` tokeniza e mascara os
datasets na **construção**. Dataset cru passado depois em `evaluate(eval_dataset=...)` chega
sem `input_ids`/`labels` e o Trainer descarta todas as colunas. Por isso os três recortes vão
como dicionário no construtor, e as métricas voltam com prefixo: `eval_geral_loss`,
`eval_humano_loss`, `eval_reescrito_loss`.

In [2]:
def prepara(caminho):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = caminho,
        max_seq_length = MAX_SEQ_LENGTH,
        load_in_4bit = LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(model)

    d = ds.map(lambda ex: {"text": [tokenizer.apply_chat_template(m, tokenize=False)
                                    for m in ex["messages"]]}, batched=True)

    # Os recortes precisam existir ANTES do construtor (ver texto acima).
    recortes = {
        "geral":     d,
        "humano":    d.filter(lambda e: not e["resposta_reescrita"]),
        "reescrito": d.filter(lambda e: e["resposta_reescrita"]),
    }

    trainer = SFTTrainer(
        model = model,
        processing_class = tokenizer,
        train_dataset = d,        # exigido pelo construtor; nunca treinamos
        eval_dataset = recortes,
        args = SFTConfig(
            output_dir = str(SAIDA / "eval_tmp"),
            seed = SEED,
            dataset_text_field = "text",
            max_length = MAX_SEQ_LENGTH,
            packing = False,
            per_device_eval_batch_size = 4,
            # Sem isso o NotebookProgressCallback quebra em evaluate() avulso:
            # ele so inicializa a tabela de metricas em on_train_begin.
            disable_tqdm = True,
            report_to = [],
        ),
    )
    return train_on_responses_only(
        trainer,
        instruction_part = "<|im_start|>user\n",
        response_part = "<|im_start|>assistant\n",
    ), recortes


def mede(trainer, recortes, rotulo):
    metricas = trainer.evaluate()      # avalia os tres recortes de uma vez
    r = {"rotulo": rotulo}
    print(f"\n=== {rotulo} ===")
    for chave, texto in [("geral", "geral"),
                         ("humano", "alvo humano"),
                         ("reescrito", "alvo reescrito por LLM")]:
        r[chave] = metricas[f"eval_{chave}_loss"]
        print(f"{texto:<24} {len(recortes[chave]):>6,} linhas | loss {r[chave]:.4f}")
    return r

In [3]:
trainer, recortes = prepara(ADAPTER)
tunado = mede(trainer, recortes, "fine-tunado")

==((====))==  Unsloth 2026.9.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Filter: 100%|██████████| 1287/1287 [00:00<00:00, 8990.01 examples/s]


{'eval_geral_loss': '1.305', 'eval_geral_model_preparation_time': '0.0304', 'eval_geral_runtime': '385', 'eval_geral_samples_per_second': '3.974', 'eval_geral_steps_per_second': '0.995', 'epoch': 0}
{'eval_humano_loss': '1.98', 'eval_humano_model_preparation_time': '0.0304', 'eval_humano_runtime': '43.29', 'eval_humano_samples_per_second': '5.614', 'eval_humano_steps_per_second': '1.409', 'epoch': 0}
{'eval_reescrito_loss': '1.203', 'eval_reescrito_model_preparation_time': '0.0304', 'eval_reescrito_runtime': '281.8', 'eval_reescrito_samples_per_second': '4.567', 'eval_reescrito_steps_per_second': '1.143', 'epoch': 0}

=== fine-tunado ===
geral                     1,530 linhas | loss 1.3049
alvo humano                 243 linhas | loss 1.9804
alvo reescrito por LLM    1,287 linhas | loss 1.2026


In [4]:
# Sem o LoRA sobra o modelo base -- mesmos pesos, mesmos dados, mesmo Trainer.
with trainer.model.disable_adapter():
    base = mede(trainer, recortes, "base (sem LoRA)")

{'eval_geral_loss': '2.216', 'eval_geral_model_preparation_time': '0.0304', 'eval_geral_runtime': '311.8', 'eval_geral_samples_per_second': '4.907', 'eval_geral_steps_per_second': '1.228', 'epoch': 0}
{'eval_humano_loss': '2.688', 'eval_humano_model_preparation_time': '0.0304', 'eval_humano_runtime': '41.83', 'eval_humano_samples_per_second': '5.809', 'eval_humano_steps_per_second': '1.458', 'epoch': 0}
{'eval_reescrito_loss': '2.148', 'eval_reescrito_model_preparation_time': '0.0304', 'eval_reescrito_runtime': '264', 'eval_reescrito_samples_per_second': '4.875', 'eval_reescrito_steps_per_second': '1.22', 'epoch': 0}

=== base (sem LoRA) ===
geral                     1,530 linhas | loss 2.2159
alvo humano                 243 linhas | loss 2.6878
alvo reescrito por LLM    1,287 linhas | loss 2.1477


## Leitura

**A primeira linha responde se valeu a pena.** Se a diferença for pequena, o fine-tuning
rendeu pouco — e aí vale reconsiderar o custo de servir um modelo próprio em vez de usar
uma API pronta.

**As duas últimas dizem o que ele aprendeu.** O ganho nas respostas reescritas por LLM é
esperado e fácil: saída de modelo é uniforme e previsível. O ganho nas respostas de médico
preservadas é o que importa — é ali que está o texto humano real.

In [5]:
rotulos = {"geral": "geral", "humano": "alvo humano",
           "reescrito": "alvo reescrito por LLM"}
print(f"{'':<24} {'base':>10} {'fine-tunado':>13} {'ganho':>10}")
for chave, texto in rotulos.items():
    b, t = base[chave], tunado[chave]
    print(f"{texto:<24} {b:>10.4f} {t:>13.4f} {(b-t)/b*100:>9.1f}%")

destino = SAIDA / f"avaliacao_{SPLIT}.json"
destino.write_text(json.dumps({"base": base, "fine_tunado": tunado},
                              indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\n-> {destino}")

del trainer
gc.collect(); torch.cuda.empty_cache()   # libera a VRAM

                               base   fine-tunado      ganho
geral                        2.2159        1.3049      41.1%
alvo humano                  2.6878        1.9804      26.3%
alvo reescrito por LLM       2.1477        1.2026      44.0%

-> ../modelos/avaliacao_val.json
